# Carregar painel processado

Carrega `panel.parquet` (gerado por `src/processing.py`) e aplica o mesmo filtro `common_start` usado na EDA.

In [ ]:
from pathlib import Path
from functools import reduce

import pandas as pd

# Carregar o mesmo painel usado na EDA
panel = pd.read_parquet("../data/processed/panel.parquet")
common_start = panel.dropna().index.min()
panel = panel.loc[common_start:]

# Converter para formato esperado pelas células downstream (coluna data_mes)
df_new = panel.reset_index().rename(columns={"date": "data_mes"})

# Salvar base consolidada
df_new.to_parquet("../data/processed/base_unica_mensal.parquet", index=False)

print(f"Shape: {df_new.shape}")
print(f"Período: {df_new['data_mes'].min().date()} -> {df_new['data_mes'].max().date()}")
print(f"Início comum: {common_start.date()}")
print(f"NaN por coluna:")
print(df_new.isnull().sum())
df_new.tail()

Shape: (167, 12)
Período: 2012-04-01 -> 2026-02-01
Início comum: 2012-04-01
NaN por coluna:
data_mes                             0
inadimplencia_pf_total               1
inadimplencia_carteira_total         1
selic_acumulada_mes                  0
ibc_br_dessaz                        2
cambio_ptax_venda                    0
ipca_variacao_mensal                 1
taxa_juros_pf_total                  1
saldo_carteira_credito_total         1
taxa_desocupacao                     2
rendimento_medio_real                2
pmc_volume_vendas_dessazonalizado    2
dtype: int64


,data_mes,inadimplencia_pf_total,inadimplencia_carteira_total,selic_acumulada_mes,ibc_br_dessaz,cambio_ptax_venda,ipca_variacao_mensal,taxa_juros_pf_total,saldo_carteira_credito_total,taxa_desocupacao,rendimento_medio_real,pmc_volume_vendas_dessazonalizado
162,2025-10-01,4.86,4.00,1.28,108.60643,5.385526,0.09,36.58,6919371.0,54.0,3543.0,10774126.0
163,2025-11-01,4.93,4.01,1.05,109.25065,5.340853,0.18,37.70,6999002.0,52.0,3581.0,10877491.0
164,2025-12-01,5.00,4.03,1.22,109.05574,5.453091,0.33,37.57,7130222.0,51.0,3613.0,10831391.0
165,2026-01-01,5.24,4.25,1.16,NaN,5.338024,0.33,37.95,7115576.0,NaN,NaN,NaN
166,2026-02-01,NaN,NaN,1.00,NaN,5.200617,NaN,NaN,NaN,NaN,NaN,NaN


# Feature Engineering

In [37]:
df = pd.read_parquet("../data/processed/base_unica_mensal.parquet")
print(df.shape)

(167, 12)


In [40]:
# Garantir ordenação e índice temporal
df["data_mes"] = pd.to_datetime(df["data_mes"])
df = df.sort_values("data_mes").reset_index(drop=True)
df = df.set_index("data_mes")
df.index.name = "data_mes"

# Checagens básicas
display(df.head())
print("Período:", df.index.min().date(), "->", df.index.max().date())
print("Frequência inferida:", pd.infer_freq(df.index))

# Missing values
na = df.isna().sum().sort_values(ascending=False)
display(na[na > 0].to_frame("n_missing"))

,inadimplencia_pf_total,inadimplencia_carteira_total,selic_acumulada_mes,ibc_br_dessaz,cambio_ptax_venda,ipca_variacao_mensal,taxa_juros_pf_total,saldo_carteira_credito_total,taxa_desocupacao,rendimento_medio_real,pmc_volume_vendas_dessazonalizado
data_mes,,,,,,,,,,,
2012-04-01,5.41,3.76,0.71,98.87951,1.854835,0.64,34.30,2106585.0,78.0,3075.0,9387725.0
2012-05-01,5.51,3.76,0.74,100.02006,1.985991,0.36,32.34,2142061.0,77.0,3067.0,9368144.0
2012-06-01,5.43,3.71,0.64,100.74394,2.049195,0.08,31.58,2175135.0,76.0,3073.0,9517799.0
2012-07-01,5.42,3.72,0.68,100.90465,2.028736,0.43,30.71,2192888.0,75.0,3089.0,9572403.0
2012-08-01,5.40,3.75,0.69,101.24114,2.029443,0.41,29.84,2219866.0,73.0,3099.0,9584401.0


Período: 2012-04-01 -> 2026-02-01
Frequência inferida: MS


,n_missing
ibc_br_dessaz,2
taxa_desocupacao,2
rendimento_medio_real,2
pmc_volume_vendas_dessazonalizado,2
inadimplencia_pf_total,1
inadimplencia_carteira_total,1
ipca_variacao_mensal,1
taxa_juros_pf_total,1
saldo_carteira_credito_total,1


Criamos transformações das séries para capturar mudanças recentes e suavizar ruído:

- **Taxas** (Selic, juros PF, desemprego, IPCA): variação mensal (Δ1m) e média móvel de 3 meses (MA3)
- **Níveis/índices** (câmbio, IBC-Br, PMC, renda, saldo de crédito): crescimento mensal (pct_change 1m) e MA3

In [49]:
df_trans = df.copy()

RATE_COLS = ["selic_acumulada_mes", "taxa_juros_pf_total", "taxa_desocupacao", "ipca_variacao_mensal"]
LEVEL_COLS = ["cambio_ptax_venda", "ibc_br_dessaz", "pmc_volume_vendas_dessazonalizado", "rendimento_medio_real", "saldo_carteira_credito_total"]

# Δ1m para taxas
for col in RATE_COLS:
    df_trans[f"{col}__d1"] = df_trans[col].diff(1)

# pct_change 1m para níveis
for col in LEVEL_COLS:
    df_trans[f"{col}__pct1"] = df_trans[col].pct_change(1)

# MA3 para todas
for col in RATE_COLS + LEVEL_COLS:
    df_trans[f"{col}__ma3"] = df_trans[col].rolling(3).mean()

print("Colunas de transformação criadas:", len(df_trans.columns) - len(df.columns))

Colunas de transformação criadas: 18


Vamos reduzir o risco de overfitting mantendo um conjunto enxuto e interpretável de features. Em séries macro mensais com ~165 observações, criar muitas transformações e muitos lags pode inflar a dimensionalidade e capturar ruído ao invés de sinal.
Por isso, vamos selecionar transformações que fazem sentido econômico e são mais estáveis:

Para variáveis do tipo taxa (Selic, taxa de juros PF, desemprego e IPCA), usaremos o nível, a variação mensal (Δ1m) e a média móvel de 3 meses (MA3), pois inadimplência tende a responder à tendência e mudanças recentes.

Para variáveis em nível/índice (câmbio, IBC-Br, PMC, renda e saldo de crédito), usaremos crescimento mensal (pct_change 1m) e MA3, para capturar dinâmica relativa e suavizar ruído.
Em seguida, aplicaremos apenas lags de curto/médio prazo (1, 3 e 6 meses) para garantir que o modelo use somente informação disponível até t−1 e reduzir dimensionalidade.

In [50]:
TARGET = "inadimplencia_pf_total"

# Features base: taxas usam nível + d1 + ma3; níveis usam pct1 + ma3
BASE_FEATURES = []

for c in RATE_COLS:
    BASE_FEATURES += [c, f"{c}__d1", f"{c}__ma3"]

for c in LEVEL_COLS:
    BASE_FEATURES += [f"{c}__pct1", f"{c}__ma3"]

# Remove duplicatas e mantém só colunas que realmente existem
BASE_FEATURES = [c for c in dict.fromkeys(BASE_FEATURES) if c in df_trans.columns]

print("N de features base (antes de lags):", len(BASE_FEATURES))
print("Features base:")
display(pd.Series(BASE_FEATURES, name="feature"))

# Criar lags dessas features
LAGS = [1, 3, 6]

df_fe = df_trans.copy()

# lags do alvo (autoregressivo)
for k in LAGS:
    df_fe[f"{TARGET}__lag{k}"] = df_fe[TARGET].shift(k)

# lags das features selecionadas
for col in BASE_FEATURES:
    for k in LAGS:
        df_fe[f"{col}__lag{k}"] = df_fe[col].shift(k)

feature_cols = [c for c in df_fe.columns if c.endswith(tuple([f"lag{k}" for k in LAGS]))]

# Dataset modelável (dropna) e separação X/y
df_model = df_fe[[TARGET] + feature_cols].dropna().copy()
X_model = df_model[feature_cols]
y_model = df_model[TARGET]

print("Após lags + dropna:")
print("X_model shape:", X_model.shape, "| y_model shape:", y_model.shape)
print("Período modelável:", df_model.index.min(), "->", df_model.index.max())

df_model.head()

N de features base (antes de lags): 22
Features base:


0                         selic_acumulada_mes
1                     selic_acumulada_mes__d1
2                    selic_acumulada_mes__ma3
3                         taxa_juros_pf_total
4                     taxa_juros_pf_total__d1
5                    taxa_juros_pf_total__ma3
6                            taxa_desocupacao
7                        taxa_desocupacao__d1
8                       taxa_desocupacao__ma3
9                        ipca_variacao_mensal
10                   ipca_variacao_mensal__d1
11                  ipca_variacao_mensal__ma3
12                    cambio_ptax_venda__pct1
13                     cambio_ptax_venda__ma3
14                        ibc_br_dessaz__pct1
15                         ibc_br_dessaz__ma3
16    pmc_volume_vendas_dessazonalizado__pct1
17     pmc_volume_vendas_dessazonalizado__ma3
18                rendimento_medio_real__pct1
19                 rendimento_medio_real__ma3
20         saldo_carteira_credito_total__pct1
21          saldo_carteira_credito

Após lags + dropna:
X_model shape: (158, 69) | y_model shape: (158,)
Período modelável: 2012-12-01 00:00:00 -> 2026-01-01 00:00:00


,inadimplencia_pf_total,inadimplencia_pf_total__lag1,inadimplencia_pf_total__lag3,inadimplencia_pf_total__lag6,selic_acumulada_mes__lag1,selic_acumulada_mes__lag3,selic_acumulada_mes__lag6,selic_acumulada_mes__d1__lag1,selic_acumulada_mes__d1__lag3,selic_acumulada_mes__d1__lag6,...,rendimento_medio_real__pct1__lag6,rendimento_medio_real__ma3__lag1,rendimento_medio_real__ma3__lag3,rendimento_medio_real__ma3__lag6,saldo_carteira_credito_total__pct1__lag1,saldo_carteira_credito_total__pct1__lag3,saldo_carteira_credito_total__pct1__lag6,saldo_carteira_credito_total__ma3__lag1,saldo_carteira_credito_total__ma3__lag3,saldo_carteira_credito_total__ma3__lag6
data_mes,,,,,,,,,,,,,,,,,,,,,
2012-12-01,5.10,5.21,5.41,5.43,0.55,0.54,0.64,-0.06,-0.15,-0.10,...,0.001956,3089.666667,3093.666667,3071.666667,0.015180,0.011085,0.015440,2.277200e+06,2.219076e+06,2.141260e+06
2013-01-01,5.03,5.10,5.34,5.42,0.55,0.61,0.68,0.00,0.07,0.04,...,0.005207,3085.000000,3093.666667,3076.333333,0.024881,0.014173,0.008162,2.318488e+06,2.246875e+06,2.170028e+06
2013-02-01,4.98,5.03,5.21,5.40,0.60,0.55,0.69,0.05,-0.06,0.01,...,0.003237,3086.000000,3089.666667,3087.000000,-0.000918,0.015180,0.012302,2.348448e+06,2.277200e+06,2.195963e+06
2013-03-01,4.94,4.98,5.10,5.41,0.49,0.55,0.54,-0.11,0.00,-0.15,...,-0.001936,3093.666667,3085.000000,3093.666667,0.007367,0.024881,0.011085,2.372700e+06,2.318488e+06,2.219076e+06
2013-04-01,4.85,4.94,5.03,5.34,0.55,0.60,0.61,0.06,0.05,0.07,...,-0.001293,3110.000000,3086.000000,3093.666667,0.018235,-0.000918,0.014173,2.392275e+06,2.348448e+06,2.246875e+06


In [25]:
df_model.to_parquet("../data/processed/base_modelagem_1.parquet", index=True)

# Base modelagem com implicações sugeridas na etapa de EDA

Nesta etapa vamos construir uma segunda versão do dataset de modelagem baseada nas conclusões da EDA. O objetivo é reduzir dimensionalidade e risco de overfitting usando (i) transformações que evitam tendência/regressão espúria (ex.: log no saldo de crédito e variações para séries em nível) e (ii) um único lag “ótimo” por variável (obtido via CCF), em vez de um grid de lags. Também incluiremos um componente autorregressivo do alvo (lag1), que se mostrou dominante, e manteremos todas as features defasadas para garantir ausência de data leakage (usar apenas informação até t−1).

In [26]:
import numpy as np
import pandas as pd

df = pd.read_parquet("../data/processed/base_unica_mensal.parquet").copy()

df["data_mes"] = pd.to_datetime(df["data_mes"])
df = df.sort_values("data_mes").set_index("data_mes")
df.index.name = "data_mes"

TARGET = "inadimplencia_pf_total"

# --------
# Transformações base (antes de lags)
# --------
df["saldo_credito_log"] = np.log(df["saldo_carteira_credito_total"])
df["cambio_pct1"] = df["cambio_ptax_venda"].pct_change(1)
df["desemprego_d1"] = df["taxa_desocupacao"].diff(1)
df["juros_pf_d3"] = df["taxa_juros_pf_total"].diff(3)

# --------
# Lags ótimos (CCF) — um lag por variável
# --------
LAG_MAP = {
    "y_lag1": (TARGET, 1),
    "selic_lag8": ("selic_acumulada_mes", 8),
    "desemprego_lag9": ("desemprego_d1", 9),
    "ibc_lag12": ("ibc_br_dessaz", 12),
    "juros_pf_lag6": ("juros_pf_d3", 6),
    "saldo_log_lag1": ("saldo_credito_log", 1),
    "inad_total_lag1": ("inadimplencia_carteira_total", 1),
    "cambio_lag1": ("cambio_pct1", 1),
}

for new_name, (col, k) in LAG_MAP.items():
    df[new_name] = df[col].shift(k)

FEATURES = list(LAG_MAP.keys())
df_model = df[[TARGET] + FEATURES].dropna().copy()

print("df_model shape:", df_model.shape)
print("Período modelável:", df_model.index.min(), "->", df_model.index.max())
df_model.head()

df_model shape: (153, 9)
Período modelável: 2013-04-01 00:00:00 -> 2025-12-01 00:00:00


,inadimplencia_pf_total,y_lag1,selic_lag8,desemprego_lag9,ibc_lag12,juros_pf_lag6,saldo_log_lag1,inad_total_lag1,cambio_lag1
data_mes,,,,,,,,,
2013-04-01,4.85,4.94,0.69,-1.0,98.87951,-1.66,14.702192,3.45,0.004860
2013-05-01,4.88,4.85,0.54,-2.0,100.02006,-1.37,14.711250,3.47,0.009771
2013-06-01,4.61,4.88,0.61,-2.0,100.74394,-2.46,14.726706,3.47,0.016297
2013-07-01,4.56,4.61,0.55,-2.0,100.90465,-1.67,14.744353,3.25,0.067874
2013-08-01,4.45,4.56,0.55,-1.0,101.24114,-0.51,14.749691,3.21,0.036455


In [27]:
df_model.to_parquet("../data/processed/base_modelagem_2.parquet", index=True)